# Sparkle Movie — Exploration & Analyse des données

**Contexte :** Nous travaillons pour une plateforme de streaming vidéo qui souhaite améliorer l'expérience utilisateur en proposant des recommandations personnalisées. Nous utilisons le dataset **MovieLens** pour analyser les comportements des utilisateurs et préparer un système de recommandation.

**Problématique :** Comment recommander des films pertinents à un utilisateur en se basant sur ses préférences passées et celles d'utilisateurs similaires ?

---

## 1. Préparation de l'environnement

### 1.1 Installation des dépendances

Les librairies nécessaires pour ce projet :

In [1]:
# Installation des librairies (à exécuter une seule fois)
# !pip install pyspark matplotlib seaborn pandas

### 1.2 Configuration de la session Spark

In [2]:
import os

# Configuration Java (nécessaire sur Windows)
os.environ["JAVA_HOME"] = "C:/Program Files/Microsoft/jdk-21.0.10.7-hotspot"

from pyspark.sql import SparkSession

# Création de la session Spark
spark = SparkSession.builder \
    .appName("SparkleMovie") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

# Réduction du niveau de log pour plus de lisibilité
spark.sparkContext.setLogLevel("WARN")

print(f"Session Spark créée avec succès !")
print(f"Version Spark : {spark.version}")
print(f"Application ID : {spark.sparkContext.applicationId}")

Session Spark créée avec succès !
Version Spark : 4.1.1
Application ID : local-1773750545703


### 1.3 Téléchargement du dataset MovieLens

Nous utilisons le dataset **MovieLens Small** (100k ratings) qui est idéal pour le développement et les tests. Il contient :
- `ratings.csv` : ~100 000 notes de 610 utilisateurs sur 9 742 films
- `movies.csv` : métadonnées de 9 742 films (titre, genres)

In [3]:
import urllib.request
import zipfile
import os

# URL du dataset MovieLens Small (100k)
MOVIELENS_URL = "https://files.grouplens.org/datasets/movielens/ml-latest-small.zip"
ZIP_PATH = "ml-latest-small.zip"
DATA_DIR = "data"

# Téléchargement si le dossier data n'existe pas encore
if not os.path.exists(DATA_DIR):
    print("Téléchargement du dataset MovieLens Small...")
    urllib.request.urlretrieve(MOVIELENS_URL, ZIP_PATH)
    print("Extraction...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extractall(".")
    os.rename("ml-latest-small", DATA_DIR)
    os.remove(ZIP_PATH)
    print(f"Dataset extrait dans le dossier '{DATA_DIR}/'")
else:
    print(f"Le dossier '{DATA_DIR}/' existe déjà.")

# Vérification du contenu
print("\nFichiers disponibles :")
for f in os.listdir(DATA_DIR):
    size = os.path.getsize(os.path.join(DATA_DIR, f))
    print(f"  {f} ({size / 1024:.1f} Ko)")

Le dossier 'data/' existe déjà.

Fichiers disponibles :
  links.csv (193.3 Ko)
  movies.csv (482.8 Ko)
  ratings.csv (2425.5 Ko)
  README.txt (8.1 Ko)
  tags.csv (115.9 Ko)


### 1.4 Vérification de l'environnement

On s'assure que toutes les librairies sont disponibles et que l'environnement est prêt.

In [4]:
import pyspark
import pandas as pd
import matplotlib
import seaborn

print("Versions des librairies :")
print(f"  PySpark   : {pyspark.__version__}")
print(f"  Pandas    : {pd.__version__}")
print(f"  Matplotlib: {matplotlib.__version__}")
print(f"  Seaborn   : {seaborn.__version__}")
print()
print("Environnement prêt !")

Versions des librairies :
  PySpark   : 4.1.1
  Pandas    : 2.3.3
  Matplotlib: 3.10.7
  Seaborn   : 0.13.2

Environnement prêt !
